### **Motivation:** The current API (CodeToCAD Legacy) works fine for quick manipulation of 3D shapes. However, 2D sketches to create custom shapes are confusing to work with. Also, querying geometry and creating constraints and assemblies leaves a lot to be desired. Lastly, some parts of the api are veeeery confusing.


1️⃣ We're keeping all the same capabilities as the current API. Especially things like using units `cube("2in", "2cm", "2in")`, will remain. In fact this particular feature will be upgraded to allow arithmetic operations in the strings, e.g. `f"2in + 2mm * {variable_with_some_dimension}"`.

In [6]:
import math

from codetocad.cad import *

In [7]:
x = Length("2mm + 1m")
assert math.isclose(x, 1.002), f"Expected 1.002m but got {x}m"

x = Length("5in")
assert math.isclose(x, 0.127), f"Expected 0.127m but got {x}m"

x = Length(f"2mm * {x}")
assert math.isclose(x, 0.000254), f"Expected 0.000254m but got {x}m"

theta = Angle("90deg + 0.5rad")
assert math.isclose(
    theta, 2.0708, abs_tol=0.0001
), f"Expected 2.0708rad but got {theta}rad"


2️⃣ Next, **naming will not be required anymore**. Previously, you had to type the name of your part: `my_beam = Part("my_beam").create_cube(1,0.2,0.2)`. Now you would use `my_beam = Assembly.add.preset.cube(1,0.2,0.2)`. You would still be able to give a name using `my_beam.set.name("beam")`Removing the requirement to name things makes lines less verbose, and removes the confusion of names clashing. _Initially, I liked forcing the naming of things so that generated documents are well styled, however, after using CodeToCAD for a few years, it simply got in the way._ Ironically, removing names also improves maintaining statelessness in CodeToCAD. Note: parts can be retrieved via `Assembly.parts[-1]` to get the last one, or `Part.get.by_name("my_beam")` or other get operators.

> Note: when I say stateless, I mean codetocad methods are **fire-and-forget**. For an active session, uuid's may be temporarily kept so that references held in variables are usable. Note: The current codetocad API supports this by using the **name** of the object to look it up again during subsequent calls.




3️⃣ I gave away the next change in the last bit. Methods will be grouped under sub-categories masquerading as attributes. e.g.`Assembly.add.preset.cube()` and `Part.set.name()` . This will reduce verbose methods such as `Part().create_cube()` and having a looong list of options in the IDE auto-complate. The idea is to make IDE and LLM auto-completion more accessible.

In [9]:
sketch = Sketch()

t = Length("5mm")
gap = Length("2mm")
sketch.draw.point("0mm", "0mm")
sketch.draw.line_to(f"10cm + 2*{t} + {gap}", "0")
sketch.draw.line_to("20cm", "10cm")

wire = sketch.get.wire(-1)
e1 = wire.get.edge(0)
e2 = wire.get.edge(1)

wire.constraint.parallel(e1, e2)

v_mid = sketch.draw.point("0", "0").v1
wire.constraint.midpoint(e1, v_mid)

print(sketch)
for e in sketch.wires:
    print(e)

<Sketch: 1 vertices, 1 edges>
Wire([Edge([0. 0. 0.], [0. 0. 0.]), Edge([0. 0. 0.], [0.112 0.    0.   ]), Edge([0. 0. 0.], [0.2 0.1 0. ]), Edge([0. 0. 0.], [0. 0. 0.])]


In [10]:
my_assembly = Assembly()
p0 = my_assembly.add.preset.cube("1", "0.2", "0.2")
p1 = my_assembly.add.preset.cube(1, 1, 1)
p2 = my_assembly.add.preset.cube(2, 2, 2)

p1.set_name("my_beam")
found = Part.get_by_name("my_beam")

print(my_assembly.get.parts)
print(my_assembly.get.part("my_beam").name)
print(my_assembly.get.part["my_beam"].name)
print(my_assembly.get.part[1].name)
print(my_assembly.get.part[-2].name)

[<Part: Unnamed. <Sketch: 1 vertices, 1 edges>>, <Part: my_beam. <Sketch: 1 vertices, 1 edges>>, <Part: Unnamed. <Sketch: 1 vertices, 1 edges>>]
my_beam
my_beam
my_beam
my_beam


4️⃣ Context based programming seems to be big recently. I don't really think it fits in the stateless nature of CodeToCAD, nor does it yield benefits here. Semantically, doing 
```
my_assembly = Assembly()
p1 = my_assembly.add.preset.cube()
p2 = my_assembly.add.preset.cube()
p1.contraint.get.edge.bottom_left.constraint.revolute(part2.get.edge.preset.top_right)
``` 
is the same as the Contextual flavor, but with less spicy syntax:
```
with Assembly() as my_assembly:
    p1 = add.preset.cube()
    p2 = add.preset.cube()
    p1.contraint.get.edge.bottom_left.constraint.revolute(part2.get.edge.preset.top_right)
```
I worry that maintaining a context means codetocad will have to remember things and/or fire off multiple commands when the context is terminated. *frowny face*. I also worry about misuse because nothing stops people from mutating other objects inside an irrelevant context, and catching that could be messy.




5️⃣ Constraints will replace _Joints_ in the current API. You can add constraints simply by doing `part1.get.edge.bottom_left.constraint.revolute(part2.get.edge.preset.top_right)`.  Similarly, sketching constraints will be `line1.constaint.parallel(line2)`.
> Note: currently, the is `Joint(part1.get_landmark(PresetLandmarks.bottom_left), part2.get_landmark(PresetLandmarks.top_right).rotation()` but it is ambiguous if you're selecting a vertex, edge or face, and it's quite verbose.

6️⃣ Multiple closed wires e.g. circles or rectangles, in the same sketch makes querying very very difficult. It's common to draw a shape within a shape, e.g. a circle with a polygon inscribed inside it :six_pointed_star: , but it's not code-CAD friendly. Most code-CAD tools out there do not deal with this, or simply assume boundaries of faces of intersecting geometry. The only way I can think of handling this is by allowing the user to query the parent group, e.g. `some_edges:list[Edge] = my_sketch.get.wires[0]` or `my_sketch.get.face[0].extrude()`. For parts it's more tricky. First of all, CAD software don't usually let you do this, but Blender does: `some_part:Part = Assembly.get.part[-1]` would work if the sub parts are not union/intersect/subtract. If they are booleaned, then the user will have to query vertices, edges and faces using Landmarks, e.g. `my_part.get.face.location(my_part.get.face.preset.top - [0,0,0.5])`, which is difficult in some scenarios, but can be avoided if parts are built carefully.
Another issue here that is not addressed in the current API is adding multiple sub_parts to a Part. `my_asembly.add.preset.cube();my_asembly.add.preset.cube()` lets you add two cubes on top of each other. No easy way to reference them except by querying the parent container, e.g. `my_asembly.parts[0]`. This is lacking in the current API.

7️⃣  Sketching in the current API is not great🦆. Trying to add sketching capabilities is what broke the current dev branch. Sketching is also a main motivation for these changes. It's a core enough feature to warrant deep-thought about our UX. There is only so much we can do with CSG, sketching is a powerful _mechanic_, pun intended.  Sketching might be done in a Contextual manner, as described in :four: . I think I will first try to implement it without.
Closed wires will form faces (can the user choose not to create a face? who knows?). This will be the goal of sketching. I imagine something similar to:
```
wire = Sketch.add.wire()
l1 = wire.add.line()
l2 = wire.add.line()
l1.constraint.parallel(l2)

Sketch.add.preset.rectangle().extrude()
```
These changes allow us to cleanly and unambigiously understand the underlying type: Part, Sketch, Wire, Edge, Vertex.